# Training: BiLSTM Models with Attention Mechanisms

This notebook trains **4 BiLSTM models** on the AG News dataset:

| # | Model |
|---|-------|
| 1 | BiLSTM (baseline) |
| 2 | BiLSTM + Additive |
| 3 | BiLSTM + Multiplicative |
| 4 | BiLSTM + Self-Attention |

**Hardware:** Colab CPU
**Dataset:** AG News (full: 120k train, 7.6k test)

Follows the training plan in `training_plan_cpu.md`.
Preprocessed data is loaded from Google Drive (run `dataprep.ipynb` first).

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/ag_news_prep'
RESULTS_DIR = '/content/drive/MyDrive/ag_news_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Data dir: {DATA_DIR}')
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
# Install dependencies
!pip install -q torch numpy scikit-learn matplotlib seaborn tqdm

In [ ]:
# Hyperparameters (from training_plan_cpu.md)
import numpy as np

VOCAB_SIZE = 20000
MAX_LEN = 48
EMBED_DIM = 100
HIDDEN_DIM = 64
ATTENTION_DIM = 32
NUM_LAYERS = 1
DROPOUT = 0.3
BATCH_SIZE = 64
MAX_EPOCHS = 8
EARLY_STOP_PATIENCE = 2
LR = 1e-3
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 1.0
SEEDS = [42, 123, 456]
NUM_CLASSES = 4

print('Hyperparameters loaded.')

In [ ]:
# Load preprocessed data
import numpy as np

X_train = np.load(f'{DATA_DIR}/X_train.npy')
y_train = np.load(f'{DATA_DIR}/y_train.npy')
X_val = np.load(f'{DATA_DIR}/X_val.npy')
y_val = np.load(f'{DATA_DIR}/y_val.npy')
X_test = np.load(f'{DATA_DIR}/X_test.npy')
y_test = np.load(f'{DATA_DIR}/y_test.npy')

import json
with open(f'{DATA_DIR}/metadata.json') as f:
    metadata = json.load(f)
with open(f'{DATA_DIR}/vocab.json') as f:
    vocab = json.load(f)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Vocab size: {len(vocab)}')
print(f'Classes: {metadata["class_names"]}')

In [ ]:
# PyTorch Dataset and DataLoader
import torch
from torch.utils.data import Dataset, DataLoader

class AGNewsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = AGNewsDataset(X_train, y_train)
val_dataset = AGNewsDataset(X_val, y_val)
test_dataset = AGNewsDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

In [ ]:
# Model definitions
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class AdditiveAttention(nn.Module):
    """Bahdanau-style additive attention"""
    def __init__(self, hidden_dim, attention_dim=32):
        super().__init__()
        self.W_h = nn.Linear(hidden_dim, attention_dim, bias=False)
        self.v = nn.Linear(attention_dim, 1, bias=False)
        self.tanh = nn.Tanh()
    
    def forward(self, lstm_out, mask=None):
        # lstm_out: [batch, seq_len, hidden_dim]
        scores = self.v(self.tanh(self.W_h(lstm_out)))  # [batch, seq_len, 1]
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(-1), float('-inf'))
        weights = F.softmax(scores, dim=1)  # [batch, seq_len, 1]
        weights = torch.nan_to_num(weights, nan=0.0)
        context = torch.sum(weights * lstm_out, dim=1)  # [batch, hidden_dim]
        return context, weights.squeeze(-1)

class MultiplicativeAttention(nn.Module):
    """Luong-style multiplicative attention"""
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_a = nn.Linear(hidden_dim, hidden_dim, bias=False)
    
    def forward(self, lstm_out, mask=None):
        # Use learned context vector as query
        query = lstm_out.mean(dim=1, keepdim=True)  # [batch, 1, hidden_dim]
        query = self.W_a(query)  # [batch, 1, hidden_dim]
        scores = torch.bmm(lstm_out, query.transpose(1, 2))  # [batch, seq_len, 1]
        scores = scores / math.sqrt(lstm_out.size(-1))
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(-1), float('-inf'))
        weights = F.softmax(scores, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        context = torch.sum(weights * lstm_out, dim=1)
        return context, weights.squeeze(-1)

class SelfAttention(nn.Module):
    """Single-head self-attention over LSTM outputs"""
    def __init__(self, hidden_dim, attention_dim=32):
        super().__init__()
        self.W_q = nn.Linear(hidden_dim, attention_dim, bias=False)
        self.W_k = nn.Linear(hidden_dim, attention_dim, bias=False)
        self.W_v = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.scale = math.sqrt(attention_dim)
    
    def forward(self, lstm_out, mask=None):
        Q = self.W_q(lstm_out)  # [batch, seq_len, attn_dim]
        K = self.W_k(lstm_out)
        V = self.W_v(lstm_out)
        
        scores = torch.bmm(Q, K.transpose(1, 2)) / self.scale  # [batch, seq_len, seq_len]
        if mask is not None:
            # mask: [batch, seq_len] -> expand to [batch, seq_len, seq_len]
            mask_expanded = mask.unsqueeze(1).expand(-1, mask.size(1), -1)
            scores = scores.masked_fill(mask_expanded, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        weights = torch.nan_to_num(weights, nan=0.0)
        attended = torch.bmm(weights, V)  # [batch, seq_len, hidden_dim]
        
        # Mean pooling over attended representations
        context = attended.mean(dim=1)  # [batch, hidden_dim]
        # Return diagonal weights for analysis
        diag_weights = torch.diagonal(weights, dim1=1, dim2=2)
        return context, diag_weights

In [ ]:
# BiLSTM model with attention
class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=1, dropout=0.3, attention_type='none', pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            bidirectional=True, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        
        lstm_out_dim = hidden_dim * 2  # bidirectional
        self.attention_type = attention_type
        
        if attention_type == 'additive':
            self.attention = AdditiveAttention(lstm_out_dim, ATTENTION_DIM)
        elif attention_type == 'multiplicative':
            self.attention = MultiplicativeAttention(lstm_out_dim)
        elif attention_type == 'self':
            self.attention = SelfAttention(lstm_out_dim, ATTENTION_DIM)
        elif attention_type == 'none':
            self.attention = None
        else:
            raise ValueError(f'Unknown attention type: {attention_type}')
        
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(lstm_out_dim, num_classes)
    
    def forward(self, x):
        # x: [batch, seq_len]
        embedded = self.embedding(x)  # [batch, seq_len, embed_dim]
        embedded = self.dropout(embedded)
        
        # Create padding mask
        mask = (x == 0)  # [batch, seq_len], True where padded
        
        lstm_out, (h_n, c_n) = self.lstm(embedded)  # [batch, seq_len, hidden_dim*2]
        
        if self.attention is not None:
            context, attn_weights = self.attention(lstm_out, mask)
        else:
            # Use last non-padded hidden state
            # Find last non-pad position for each sequence
            lengths = (~mask).sum(dim=1)  # [batch]
            last_idx = (lengths - 1).clamp(min=0)
            batch_idx = torch.arange(x.size(0), device=x.device)
            context = lstm_out[batch_idx, last_idx]  # [batch, hidden_dim*2]
            attn_weights = None
        
        context = self.dropout(context)
        logits = self.classifier(context)
        return logits, attn_weights

def create_bilstm_model(attention_type='none'):
    return BiLSTMAttention(
        vocab_size=VOCAB_SIZE,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        num_classes=NUM_CLASSES,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        attention_type=attention_type,
        pad_idx=0
    )

print('Model definitions loaded.')

In [ ]:
# Training utilities
import torch
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import accuracy_score, f1_score, classification_report
import time
import json
import random
import numpy as np

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(texts)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        logits, _ = model(texts)
        loss = criterion(logits, labels)
        
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    acc = correct / total
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / total, acc, macro_f1, all_preds, all_labels

print('Training utilities loaded.')

In [ ]:
# Training loop with early stopping
def train_model(attention_type, seed, epochs=MAX_EPOCHS):
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = create_bilstm_model(attention_type).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
    
    best_val_loss = float('inf')
    best_val_f1 = 0
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    
    start_time = time.time()
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        
        scheduler.step(val_f1)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        
        # Early stopping on validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), f'{RESULTS_DIR}/best_bilstm_{attention_type}_seed{seed}.pt')
        else:
            patience_counter += 1
        
        print(f'  Epoch {epoch+1}/{epochs}: Train Loss={train_loss:.4f} Acc={train_acc:.4f} | Val Loss={val_loss:.4f} Acc={val_acc:.4f} F1={val_f1:.4f} | Patience={patience_counter}/{EARLY_STOP_PATIENCE}')
        
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break
    
    train_time = time.time() - start_time
    
    # Load best checkpoint and evaluate on test set
    model.load_state_dict(torch.load(f'{RESULTS_DIR}/best_bilstm_{attention_type}_seed{seed}.pt', map_location=device))
    test_loss, test_acc, test_f1, preds, labels = evaluate(model, test_loader, criterion, device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    from sklearn.metrics import classification_report
    report = classification_report(labels, preds, output_dict=True,
                                    target_names=['World', 'Sports', 'Business', 'Sci/Tech'])
    
    result = {
        'model': f'BiLSTM-{attention_type}',
        'attention_type': attention_type,
        'bidirectional': True,
        'seed': seed,
        'test_accuracy': float(test_acc),
        'test_macro_f1': float(test_f1),
        'best_val_f1': float(best_val_f1),
        'total_params': total_params,
        'trainable_params': trainable_params,
        'train_time_sec': train_time,
        'epochs_trained': len(history['train_loss']),
        'history': history,
        'classification_report': report
    }
    
    # Save result immediately (so disconnect loses only current run)
    result_path = f'{RESULTS_DIR}/result_bilstm_{attention_type}_seed{seed}.json'
    with open(result_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'  Result saved to {result_path}')
    
    return result

print('Training function defined.')

In [ ]:
# Run all 4 BiLSTM models × 3 seeds
# Skip any run that already has a saved result
import os

MODELS = ['none', 'additive', 'multiplicative', 'self']
SEEDS = [42, 123, 456]

all_results = []
for attn_type in MODELS:
    for seed in SEEDS:
        result_path = f'{RESULTS_DIR}/result_bilstm_{attn_type}_seed{seed}.json'
        if os.path.exists(result_path):
            print(f'Skipping {attn_type} seed={seed} (already done)')
            with open(result_path) as f:
                all_results.append(json.load(f))
            continue
        
        print(f'\n{"="*60}')
        print(f'Training: BiLSTM-{attn_type} | Seed={seed}')
        print(f'{"="*60}')
        result = train_model(attn_type, seed)
        all_results.append(result)
        
        # Print summary
        print(f'\n  >>> Test Acc: {result["test_accuracy"]:.4f} | Test F1: {result["test_macro_f1"]:.4f} | Params: {result["trainable_params"]:,}')

print('\n\nAll experiments complete!')

In [ ]:
# Results summary table
import pandas as pd
import numpy as np

summary_data = []
for r in all_results:
    summary_data.append({
        'Model': r['model'],
        'Seed': r['seed'],
        'Test Acc': r['test_accuracy'],
        'Test Macro F1': r['test_macro_f1'],
        'Params': r['trainable_params'],
        'Train Time (s)': r['train_time_sec'],
        'Epochs': r['epochs_trained']
    })

df = pd.DataFrame(summary_data)
print('\n=== Individual Results ===')
print(df.to_string(index=False))

# Aggregate by model
agg = df.groupby('Model').agg({
    'Test Acc': ['mean', 'std'],
    'Test Macro F1': ['mean', 'std'],
    'Params': 'first',
    'Train Time (s)': 'mean',
    'Epochs': 'mean'
}).round(4)

print('\n=== Aggregated Results (Mean ± Std) ===')
print(agg.to_string())

df.to_csv(f'{RESULTS_DIR}/all_results.csv', index=False)
print(f'\nResults saved to {RESULTS_DIR}/all_results.csv')

In [ ]:
# Per-class F1 comparison
import matplotlib.pyplot as plt
import seaborn as sns

class_names = ['World', 'Sports', 'Business', 'Sci/Tech']
models = ['none', 'additive', 'multiplicative', 'self']
model_names = ['BiLSTM', 'BiLSTM+Add', 'BiLSTM+Mult', 'BiLSTM+Self']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (attn, name) in enumerate(zip(models, model_names)):
    seed_results = [r for r in all_results if r['attention_type'] == attn]
    if not seed_results:
        continue
    # Average per-class F1 across seeds
    f1s = np.zeros((len(class_names), len(seed_results)))
    for j, r in enumerate(seed_results):
        for k, c in enumerate(class_names):
            f1s[k, j] = r['classification_report'][c]['f1-score']
    
    sns.heatmap(f1s, annot=True, fmt='.3f', xticklabels=[f'Seed {r["seed"]}' for r in seed_results],
                yticklabels=class_names, ax=axes[i], cmap='YlGnBu', vmin=0.8, vmax=1.0)
    axes[i].set_title(name)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_class_f1.png', dpi=300, bbox_inches='tight')
plt.show()
print('Per-class F1 heatmap saved.')

In [ ]:
# Accuracy and F1 comparison bar charts
import matplotlib.pyplot as plt
import numpy as np

model_labels = ['BiLSTM', 'BiLSTM+Add', 'BiLSTM+Mult', 'BiLSTM+Self']
attn_types = ['none', 'additive', 'multiplicative', 'self']

acc_means = []
acc_stds = []
f1_means = []
f1_stds = []

for attn in attn_types:
    subset = [r for r in all_results if r['attention_type'] == attn]
    accs = [r['test_accuracy'] for r in subset]
    f1s = [r['test_macro_f1'] for r in subset]
    acc_means.append(np.mean(accs))
    acc_stds.append(np.std(accs))
    f1_means.append(np.mean(f1s))
    f1_stds.append(np.std(f1s))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(len(model_labels))
width = 0.6

ax1.bar(x, acc_means, width, yerr=acc_stds, capsize=5, color=['#4C72B0', '#55A868', '#C44E52', '#8172B3'])
ax1.set_ylabel('Test Accuracy')
ax1.set_title('Test Accuracy (Mean ± Std)')
ax1.set_xticks(x)
ax1.set_xticklabels(model_labels, rotation=15)
ax1.set_ylim(0.88, 0.95)
for i, v in enumerate(acc_means):
    ax1.text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9)

ax2.bar(x, f1_means, width, yerr=f1_stds, capsize=5, color=['#4C72B0', '#55A868', '#C44E52', '#8172B3'])
ax2.set_ylabel('Test Macro F1')
ax2.set_title('Test Macro F1 (Mean ± Std)')
ax2.set_xticks(x)
ax2.set_xticklabels(model_labels, rotation=15)
ax2.set_ylim(0.88, 0.95)
for i, v in enumerate(f1_means):
    ax2.text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/accuracy_f1_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Comparison charts saved.')

In [ ]:
# Parameter count and training time comparison
import matplotlib.pyplot as plt
import numpy as np

model_labels = ['BiLSTM', 'BiLSTM+Add', 'BiLSTM+Mult', 'BiLSTM+Self']
attn_types = ['none', 'additive', 'multiplicative', 'self']

params = []
times = []
for attn in attn_types:
    subset = [r for r in all_results if r['attention_type'] == attn]
    params.append(subset[0]['trainable_params'])
    times.append(np.mean([r['train_time_sec'] for r in subset]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B3']
x = np.arange(len(model_labels))

ax1.bar(x, params, color=colors)
ax1.set_ylabel('Trainable Parameters')
ax1.set_title('Model Size')
ax1.set_xticks(x)
ax1.set_xticklabels(model_labels, rotation=15)
for i, v in enumerate(params):
    ax1.text(i, v + 1000, f'{v:,}', ha='center', fontsize=9)

ax2.bar(x, times, color=colors)
ax2.set_ylabel('Training Time (seconds)')
ax2.set_title('Training Time (Mean per Seed)')
ax2.set_xticks(x)
ax2.set_xticklabels(model_labels, rotation=15)
for i, v in enumerate(times):
    ax2.text(i, v + 5, f'{v:.1f}s', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/params_time_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Parameter and time comparison saved.')

In [ ]:
# Statistical significance tests
from scipy import stats
import numpy as np

baseline = [r for r in all_results if r['attention_type'] == 'none']
baseline_acc = [r['test_accuracy'] for r in baseline]
baseline_f1 = [r['test_macro_f1'] for r in baseline]

print('=== Paired t-tests vs BiLSTM Baseline ===')
for attn in ['additive', 'multiplicative', 'self']:
    variant = [r for r in all_results if r['attention_type'] == attn]
    variant_acc = [r['test_accuracy'] for r in variant]
    variant_f1 = [r['test_macro_f1'] for r in variant]
    
    t_acc, p_acc = stats.ttest_ind(baseline_acc, variant_acc)
    t_f1, p_f1 = stats.ttest_ind(baseline_f1, variant_f1)
    
    # Cohen's d
    def cohens_d(x, y):
        nx, ny = len(x), len(y)
        dof = nx + ny - 2
        pooled_std = np.sqrt(((nx-1)*np.var(x, ddof=1) + (ny-1)*np.var(y, ddof=1)) / dof)
        return (np.mean(x) - np.mean(y)) / pooled_std
    
    d_acc = cohens_d(variant_acc, baseline_acc)
    d_f1 = cohens_d(variant_f1, baseline_f1)
    
    print(f'\nBiLSTM-{attn} vs BiLSTM:')
    print(f'  Accuracy: t={t_acc:.3f}, p={p_acc:.4f}, Cohen d={d_acc:.3f}')
    print(f'  Macro F1: t={t_f1:.3f}, p={p_f1:.4f}, Cohen d={d_f1:.3f}')
    
    if p_acc < 0.05:
        print(f'  -> Statistically significant improvement in accuracy (p < 0.05)')
    else:
        print(f'  -> No statistically significant difference in accuracy (p >= 0.05)')

In [ ]:
# Attention heatmap visualization for sample sentences
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Load a model and inspect attention weights
attn_type = 'additive'  # Change to 'multiplicative' or 'self'
model = create_bilstm_model(attn_type)
model.load_state_dict(torch.load(f'{RESULTS_DIR}/best_bilstm_{attn_type}_seed42.pt', map_location='cpu'))
model.eval()

# Load vocab for decoding
idx2word = {v: k for k, v in vocab.items()}

def decode_sequence(ids):
    return [idx2word.get(i, '<unk>') for i in ids if i != 0]

# Pick a few test samples
sample_indices = [0, 1, 2, 3, 4]
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']

for idx in sample_indices:
    text_ids = torch.tensor(X_test[idx:idx+1], dtype=torch.long)
    label = y_test[idx]
    
    with torch.no_grad():
        logits, attn_weights = model(text_ids)
        pred = logits.argmax(dim=1).item()
    
    tokens = decode_sequence(X_test[idx])
    weights = attn_weights.squeeze().cpu().numpy()
    weights = weights[:len(tokens)]  # trim padding
    
    fig, ax = plt.subplots(figsize=(14, 3))
    sns.barplot(x=list(range(len(weights))), y=weights, ax=ax, palette='Blues_d')
    ax.set_xticks(range(len(weights)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Attention Weight')
    ax.set_title(f'{attn_type.capitalize()} Attention | True: {class_names[label]} | Pred: {class_names[pred]}')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/attention_{attn_type}_sample{idx}.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close()

print('Attention visualizations saved.')

## Summary

This notebook trains and evaluates 4 BiLSTM models on the AG News dataset:

1. **BiLSTM (baseline)** - No attention, uses last hidden state
2. **BiLSTM + Additive** - Bahdanau-style attention
3. **BiLSTM + Multiplicative** - Luong-style attention
4. **BiLSTM + Self-Attention** - Single-head self-attention

**Key outputs saved to Google Drive:**
- `result_bilstm_{type}_seed{seed}.json` - Individual run results
- `best_bilstm_{type}_seed{seed}.pt` - Best model checkpoints
- `all_results.csv` - Aggregated results table
- `per_class_f1.png` - Per-class F1 heatmaps
- `accuracy_f1_comparison.png` - Accuracy and F1 bar charts
- `params_time_comparison.png` - Parameter count and training time
- `attention_{type}_sample{idx}.png` - Attention weight visualizations

**For your paper:**
- Use the summary table for the main results section
- Use per-class F1 heatmaps for class-specific analysis
- Use attention visualizations for the interpretability discussion
- Use statistical test results for significance claims